<a href="https://colab.research.google.com/github/lelongc/rac/blob/main/Qwen_3_TTS_By.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title ⚡️ SPEED PODCAST STUDIO (Khôi phục bản chuẩn)
import os

# --- 1. CÀI ĐẶT THƯ VIỆN (Chỉ chạy 1 lần) ---
print("⏳ Đang cài đặt thư viện (Qwen-TTS, Flash-Attn, Pydub)...")
os.system('pip install -U qwen-tts gradio huggingface_hub pydub')
os.system('pip install flash-attn --no-build-isolation')
os.system('apt-get install -y ffmpeg')

import gradio as gr
from qwen_tts import Qwen3TTSModel
import torch
import soundfile as sf
import tempfile
import gc
import re
from pydub import AudioSegment

# ================= QUẢN LÝ MODEL & BỘ NHỚ =================
# Tối ưu hóa GPU
torch.backends.cudnn.benchmark = True
current_model = None
current_model_type = None

def load_model_smart(task_type):
    global current_model, current_model_type

    # Chọn model dựa trên task
    if task_type == "DESIGN":
        model_name = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
    else: # CLONE (Dùng cho Podcast để clone giọng Nam/Nữ chuẩn)
        model_name = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"

    # Nếu model đã tải rồi thì dùng lại
    if current_model_type == task_type and current_model is not None:
        return current_model

    # Nếu đang tải model khác thì xóa đi để giải phóng VRAM
    if current_model:
        del current_model
        gc.collect()
        torch.cuda.empty_cache()

    print(f"📥 Đang tải Model {task_type} ({model_name})...")
    try:
        # Load model với cấu hình tối ưu cho T4
        current_model = Qwen3TTSModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="cuda:0",
            attn_implementation="sdpa" # Dùng SDPA an toàn hơn FlashAttn trên Colab Free
        )
        current_model_type = task_type
        print("✅ Model Ready!")
        return current_model
    except Exception as e:
        print(f"❌ Lỗi tải model: {e}")
        return None

# ================= TAB 1: TẠO MẪU GIỌNG (Voice Design) =================
def create_voice_sample(prompt_text, sample_text):
    model = load_model_smart("DESIGN")
    if not model: return None

    print(f"🎨 Đang thiết kế giọng: {prompt_text[:30]}...")
    with torch.inference_mode():
        # Tạo giọng từ mô tả văn bản
        w, sr = model.generate_voice_design(text=sample_text, instruct=prompt_text)

    temp_wav = tempfile.NamedTemporaryFile(delete=False, suffix=".wav").name
    sf.write(temp_wav, w[0], sr)
    return temp_wav

# ================= TAB 2: PODCAST SIÊU TỐC (Voice Clone) =================
def generate_podcast_fast(script_text, nam_ref_path, nu_ref_path):
    if not script_text or not nam_ref_path or not nu_ref_path:
        print("⚠️ Thiếu kịch bản hoặc file giọng mẫu!")
        return None

    model = load_model_smart("CLONE")

    # Chuẩn bị file âm thanh nền
    final_audio = AudioSegment.silent(duration=500) # 0.5s đầu
    gap = AudioSegment.silent(duration=400) # 0.4s nghỉ giữa các câu

    lines = script_text.strip().split('\n')
    total_lines = len([l for l in lines if ":" in l])

    # --- BƯỚC QUAN TRỌNG: TÍNH TOÁN MAP GIỌNG 1 LẦN (Pre-calculate) ---
    print("⚡️ Đang phân tích file mẫu Nam & Nữ (Chỉ làm 1 lần)...")

    try:
        # Tính feature giọng Nam
        nam_prompt_feature = model.create_voice_clone_prompt(
            ref_audio=nam_ref_path, ref_text=None, x_vector_only_mode=True
        )
        # Tính feature giọng Nữ
        nu_prompt_feature = model.create_voice_clone_prompt(
            ref_audio=nu_ref_path, ref_text=None, x_vector_only_mode=True
        )
        print("✅ Đã học xong giọng! Bắt đầu render...")
    except Exception as e:
        print(f"❌ Lỗi đọc file mẫu: {e}")
        return None

    # --- VÒNG LẶP TẠO AUDIO ---
    for index, line in enumerate(lines):
        if ":" in line:
            name_part, text = line.split(":", 1)
            name_part = name_part.strip()
            text = text.strip()

            # Chuẩn hóa tên để nhận diện Nam/Nữ
            clean_name = re.sub(r'\(.*?\)', '', name_part).strip().lower()
            is_nam = clean_name in ["nam", "man", "male", "host", "teacher", "eric", "ryan", "mr"]

            # Chọn feature map tương ứng (Không cần load lại file wav)
            current_prompt = nam_prompt_feature if is_nam else nu_prompt_feature

            if text:
                print(f"🎙️ [{index+1}/{total_lines}] {clean_name.upper()}: {text[:30]}...")

                try:
                    with torch.inference_mode():
                        w, sr = model.generate_voice_clone(
                            text=text,
                            voice_clone_prompt=current_prompt # Dùng map đã tính -> Cực nhanh
                        )

                    # Lưu tạm và ghép nối
                    temp_wav = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
                    sf.write(temp_wav.name, w[0], sr)

                    segment = AudioSegment.from_wav(temp_wav.name)
                    final_audio += segment + gap

                    # Dọn dẹp file tạm
                    os.unlink(temp_wav.name)

                except Exception as e:
                    print(f"⚠️ Lỗi dòng này: {e}")

    # Xuất file cuối
    output_path = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3").name
    final_audio.export(output_path, format="mp3")
    print("✅ HOÀN TẤT! File Podcast đã xong.")
    return output_path

# ================= GIAO DIỆN (Giữ nguyên cấu trúc cũ) =================
css = """
body{background: #0b0f19; color: #e5e7eb;}
.panel{background: #1f2937; padding: 20px; border-radius: 12px;}
"""

with gr.Blocks(title="Speed Podcast Studio", css=css, theme=gr.themes.Base()) as demo:
    gr.Markdown("# ⚡️ SPEED PODCAST STUDIO (Khôi Phục)")
    gr.Markdown("Quy trình: **B1. Tạo giọng mẫu** -> **B2. Tải về máy** -> **B3. Upload lại vào B2 để render Podcast**.")

    with gr.Tab("B1: TẠO MẪU GIỌNG (Voice Design)"):
        with gr.Row():
            with gr.Column(elem_classes="panel"):
                gr.Markdown("### 👨 Tạo Mẫu Nam")
                nam_p = gr.Textbox(label="Mô tả giọng Nam", value="A professional male podcast host, deep and soothing voice. Laughing and energetic.")
                nam_s = gr.Textbox(label="Câu nói mẫu", value="Haha! Hello everyone, welcome to the show.")
                b_nam = gr.Button("Tạo & Nghe Thử")
                o_nam = gr.Audio(label="Kết quả Nam (Tải về nếu ưng ý)", type="filepath")

            with gr.Column(elem_classes="panel"):
                gr.Markdown("### 👩 Tạo Mẫu Nữ")
                nu_p = gr.Textbox(label="Mô tả giọng Nữ", value="A warm female voice, soft and velvety. American accent. Very happy.")
                nu_s = gr.Textbox(label="Câu nói mẫu", value="Wow! I am so excited to be here.")
                b_nu = gr.Button("Tạo & Nghe Thử")
                o_nu = gr.Audio(label="Kết quả Nữ (Tải về nếu ưng ý)", type="filepath")

    with gr.Tab("B2: RENDER PODCAST (Voice Clone)"):
        gr.Markdown("Upload file giọng mẫu (đã tạo ở B1 hoặc file có sẵn) vào đây để Clone.")
        with gr.Row():
            with gr.Column(elem_classes="panel"):
                r_nam = gr.Audio(label="Upload File Mẫu Nam", type="filepath")
                r_nu = gr.Audio(label="Upload File Mẫu Nữ", type="filepath")

            with gr.Column(elem_classes="panel"):
                script = gr.Textbox(lines=10, label="Kịch bản (Nam: ... / Nu: ...)",
                                  value="Nam: Haha! Welcome back.\nNu: Wow! This is fast.\nNam: Yes it is.")
                btn = gr.Button("🚀 RENDER SIÊU TỐC", variant="primary")
                out = gr.Audio(label="Kết quả Podcast")

    # Sự kiện
    b_nam.click(create_voice_sample, [nam_p, nam_s], [o_nam])
    b_nu.click(create_voice_sample, [nu_p, nu_s], [o_nu])

    # Render Podcast
    btn.click(generate_podcast_fast, [script, r_nam, r_nu], [out])

print("🌐 Đang khởi chạy...")
demo.launch(share=True, debug=True)

⏳ Đang cài đặt thư viện (Qwen-TTS, Flash-Attn, Pydub)...



    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    
/tmp/ipython-input-1295489905.py:155: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Speed Podcast Studio", css=css, theme=gr.themes.Base()) as demo:


🌐 Đang khởi chạy...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://db9a8ad9fb771ccaa9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


📥 Đang tải Model DESIGN (Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

🎙️ [2/4] NU: Wow! You sound so happy Eric. ...


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


🎙️ [3/4] NAM: I finally found this AI tool. ...


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


🎙️ [4/4] NU: Hhh... Thank god! Finally, we ...
✅ HOÀN TẤT! File Podcast đã xong.
